In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

class GNN_Expert_Model(torch.nn.Module):
    def __init__(self):
        super(GNN_Expert_Model, self).__init__()
        
        self.conv1 = GCNConv(dataset.num_node_features, 16)
       
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)

        
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN_Expert_Model().to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f'Epoch {epoch}: Loss {loss.item():.4f}')
  
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Final Test Accuracy: {acc:.4f}')

Epoch 0: Loss 1.9440
Epoch 20: Loss 0.2790
Epoch 40: Loss 0.0622
Epoch 60: Loss 0.0415
Epoch 80: Loss 0.0386
Epoch 100: Loss 0.0499
Epoch 120: Loss 0.0328
Epoch 140: Loss 0.0250
Epoch 160: Loss 0.0340
Epoch 180: Loss 0.0648
Final Test Accuracy: 0.7990
